# 01 — Data Acquisition (MIMIC-IV Demo)

This notebook downloads the open-access **MIMIC-IV Clinical Database Demo (v2.2)**, loads its relational tables, and reshapes them into the **canonical hourly time-series** used by the rest of the pipeline.

> Replaces the former PhysioNet-2019 flat-file loader. The 2019 Challenge set has no urine output, which makes full KDIGO AKI staging impossible. The MIMIC-IV demo has `icu/outputevents`, so we can build the complete creatinine **and** urine-output criteria downstream.

The demo is **de-identified and open-access** — no PhysioNet credentialing is required (that gate applies only to the full MIMIC-IV).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q pandas numpy pyarrow tqdm scipy scikit-learn xgboost lightgbm shap matplotlib seaborn imbalanced-learn optuna boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 140.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.6 MB/s eta 0:00:00


## Download the MIMIC-IV Demo

The demo is mirrored on the **PhysioNet open-data S3 bucket** (`physionet-open/mimic-iv-demo/`) and is downloaded anonymously — no AWS account or PhysioNet login needed. The cell below is idempotent: it skips any file already present, so it is safe to re-run.

The data lands in `../data/raw/mimic_iv_demo/`, preserving the `hosp/` and `icu/` folder structure that `data_loader` expects.

In [4]:
import os, shutil, time
SRC = "/content/drive/MyDrive/sentinel-poc/project_sentinel/data/raw/mimic_iv_full/physionet.org/files/mimiciv/3.1"
DST = "/content/mimic_local"
os.makedirs(f"{DST}/hosp", exist_ok=True)
os.makedirs(f"{DST}/icu", exist_ok=True)

needed = [
    ("icu",  "chartevents.csv.gz"),
    ("icu",  "outputevents.csv.gz"),
    ("icu",  "icustays.csv.gz"),
    ("hosp", "labevents.csv.gz"),
    ("hosp", "patients.csv.gz"),
    ("hosp", "prescriptions.csv.gz"),
    ("hosp", "microbiologyevents.csv.gz"),
]
for sub, fn in needed:
    s, d = f"{SRC}/{sub}/{fn}", f"{DST}/{sub}/{fn}"
    t0 = time.time()
    print(f"copying {sub}/{fn} ...", end=" ", flush=True)
    shutil.copy(s, d)
    print(f"done  {os.path.getsize(d)/1e6:.0f} MB  ({time.time()-t0:.0f}s)")
print("✓ ALL COPIED to", DST)

copying icu/chartevents.csv.gz ... done  3502 MB  (40s)
copying icu/outputevents.csv.gz ... done  49 MB  (3s)
copying icu/icustays.csv.gz ... done  3 MB  (0s)
copying hosp/labevents.csv.gz ... done  2593 MB  (33s)
copying hosp/patients.csv.gz ... done  3 MB  (1s)
copying hosp/prescriptions.csv.gz ... done  606 MB  (8s)
copying hosp/microbiologyevents.csv.gz ... done  118 MB  (2s)
✓ ALL COPIED to /content/mimic_local


In [5]:
import os, sys
# point this at YOUR clone location if different
PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
os.chdir(os.path.join(PROJ, 'notebooks'))   # so ../data/... paths resolve
sys.path.insert(0, PROJ)                      # so `from src...` works
print('cwd =', os.getcwd())

cwd = /content/drive/MyDrive/sentinel-poc/project_sentinel/notebooks


In [ ]:
# DISABLED for full-MIMIC-IV runs. This cell downloaded the 100-patient DEMO
# from the PhysioNet open S3 bucket. For full-data training the copy cell above
# (Drive -> /content/mimic_local) supplies the data instead. Do not run this.
#
# To fall back to the demo, restore from git history and set
# data_dir = "../data/raw/mimic_iv_demo" in the cell below.
print("Skipped: demo download disabled (using full MIMIC-IV from /content/mimic_local).")

In [7]:
import sys
from pathlib import Path

# Add the project root to sys.path so we can import from src
sys.path.append(str(Path.cwd().parent))

from src.data_loader import (
    load_mimic_demo,
    build_hourly_cohort,
    identify_sepsis_onset,
    print_data_summary,
)

data_dir = "/content/mimic_local"
print(f"Data directory set to: {data_dir}")

print("Loading MIMIC-IV demo tables...")
tables = load_mimic_demo(data_dir)
for name, t in tables.items():
    print(f"  {name:20s} {len(t):>9,} rows")

Data directory set to: /content/mimic_local
Loading MIMIC-IV demo tables...
  patients               364,627 rows
  icustays                94,458 rows
  outputevents         5,359,395 rows
  prescriptions        20,292,611 rows
  microbiologyevents   3,988,224 rows


In [8]:
import pandas as pd

print("Building hourly time-series cohort (one row per ICU stay-hour)...")
hourly = build_hourly_cohort(tables, data_dir)

print("\nIdentifying sepsis onset (Sepsis-3 suspicion-of-infection proxy)...")
hourly = identify_sepsis_onset(hourly, tables)

print()
print_data_summary(hourly)
display(hourly.head())

# Persist the raw hourly frame for the cohort-definition notebook (02)
interim = Path("../data/interim")
interim.mkdir(parents=True, exist_ok=True)
hourly.to_parquet(interim / "hourly_cohort.parquet", index=False)
print(f"\nSaved → {interim / 'hourly_cohort.parquet'}  (shape={hourly.shape})")

Building hourly time-series cohort (one row per ICU stay-hour)...

Identifying sepsis onset (Sepsis-3 suspicion-of-infection proxy)...

--------------------------------------------
MIMIC-IV DEMO COHORT SUMMARY
--------------------------------------------
ICU stays:          94,458
Patient-hours:      8,275,264
Septic stays:       29,700
Creatinine values:  573,853
Hours w/ urine:     4,050,233 (48.9%)
--------------------------------------------


,stay_id,ICULOS,DBP,HR,MAP,O2Sat,Resp,SBP,Temp,AST,...,pH,urine_rate,urine_observed,weight_kg,subject_id,Age,Gender,patient_id,hospital_system,t_sepsis
0,30000153,0,77.0,104.0,84.0,100.0,17.000000,113.0,NaN,NaN,...,NaN,3.916084,1,71.5,12466550,61,1,30000153,A,NaN
1,30000153,1,66.5,NaN,80.0,NaN,16.000000,141.0,37.277778,NaN,...,7.30,0.629371,1,71.5,12466550,61,1,30000153,A,NaN
2,30000153,2,NaN,83.0,NaN,100.0,NaN,NaN,NaN,NaN,...,NaN,0.699301,1,71.5,12466550,61,1,30000153,A,NaN
3,30000153,3,60.0,87.5,77.5,100.0,14.666667,116.0,37.500000,NaN,...,7.31,0.699301,1,71.5,12466550,61,1,30000153,A,NaN
4,30000153,4,56.0,103.0,71.0,100.0,20.000000,111.0,NaN,NaN,...,NaN,0.629371,1,71.5,12466550,61,1,30000153,A,NaN



Saved → ../data/interim/hourly_cohort.parquet  (shape=(8275264, 41))


In [9]:
import os, glob
root = "../data/raw/mimic_iv_full/physionet.org/files/mimiciv/3.1"

for rel in ["icu/chartevents.csv.gz", "hosp/labevents.csv.gz", "icu/outputevents.csv.gz"]:
    p = f"{root}/{rel}"
    print(f"{rel:35s} exists={os.path.exists(p)}  size={os.path.getsize(p) if os.path.exists(p) else 'N/A'}")

print("\nglob chartevents:", glob.glob(f"{root}/**/chartevents.csv.gz", recursive=True))
print("glob labevents: ", glob.glob(f"{root}/**/labevents.csv.gz", recursive=True))

print("\nicu/ dir contents:")
print(sorted(os.listdir(f"{root}/icu")))
print("\nhosp/ has labevents?:", "labevents.csv.gz" in os.listdir(f"{root}/hosp"))

icu/chartevents.csv.gz              exists=True  size=3502392765
hosp/labevents.csv.gz               exists=True  size=2592909134
icu/outputevents.csv.gz             exists=True  size=49307639

glob chartevents: ['../data/raw/mimic_iv_full/physionet.org/files/mimiciv/3.1/icu/chartevents.csv.gz']
glob labevents:  ['../data/raw/mimic_iv_full/physionet.org/files/mimiciv/3.1/hosp/labevents.csv.gz']

icu/ dir contents:
['caregiver.csv.gz', 'chartevents.csv.gz', 'd_items.csv.gz', 'datetimeevents.csv.gz', 'icustays.csv.gz', 'index.html', 'ingredientevents.csv.gz', 'inputevents.csv.gz', 'outputevents.csv.gz', 'procedureevents.csv.gz']

hosp/ has labevents?: True


## Conclusion

The MIMIC-IV demo has been downloaded, its relational tables loaded, and reshaped into the canonical hourly time-series (`hourly_cohort.parquet`) — including the **urine-output rate and its observation mask**, which the downstream KDIGO labeling relies on.

**Next:** `02_cohort_definition.ipynb` applies inclusion criteria, computes baseline creatinine, and excludes prevalent AKI to produce the final modeling cohort.

> ⚠️ Sepsis onset currently uses the *suspicion-of-infection* half of Sepsis-3 (antibiotic + culture pairing). The SOFA ≥ 2 component is a documented simplification for the demo — see `identify_sepsis_onset` in `src/data_loader.py`.

In [ ]:
!git config --global user.email "dcsenga442@gmail.com"
!git config --global user.name  "Sng43"

from getpass import getpass
tok = getpass("GitHub PAT: ")
# We use the token to set the remote, then immediately clear it from memory
!cd /content/drive/MyDrive/sentinel-poc && git remote set-url origin https://{tok}@github.com/Sng43/sentinel-poc.git

# Clear the variable and the cell output manually to ensure it isn't saved in the .ipynb
tok = None
print("Remote updated. Token cleared from memory.")

In [ ]:
!git pull

### Verify Git Configuration
Run the cell below to check if your remote URL now correctly includes your token (masked) and that the project path is valid.

In [ ]:
%cd /content/drive/MyDrive/sentinel-poc

# 1. Reconcile with the remote using rebase
# This brings in any remote changes and puts your clean work on top
print("Reconciling history...")
!git pull --rebase origin main

# 2. Add and commit your current clean state
!git add -A
!git commit -m "run: 01_data_acquisition (clean)"

# 3. Force push to overwrite the 'bad' history on GitHub with the 'clean' one
print("Pushing clean history to GitHub...")
!git push origin main --force